# 04b - TabFM JAX Modeling

This notebook evaluates Google's TabFM tabular foundation model with the JAX backend in WSL. It mirrors `04a_tabFM_modeling.ipynb` but uses the JAX/Orbax checkpoint from Hugging Face and should be run with the WSL kernel `Python (churn_ml_wsl_env001)`.

Native Windows JAX is CPU-only. In WSL2, JAX can see the NVIDIA GPU, but the TabFM JAX checkpoint restore has exceeded the 8 GB VRAM available on the RTX 4060 Laptop GPU in this project. The notebook therefore defaults JAX to CPU so it can run reliably. Use `04a_tabFM_modeling.ipynb` for the GPU-backed TabFM comparison against XGBoost.


In [ ]:
# Configure JAX before importing it. Restart the kernel after changing these settings.
import os

JAX_EXECUTION_PLATFORM = os.getenv("CHURN_TABFM_JAX_PLATFORM", "cpu").strip().lower()
if JAX_EXECUTION_PLATFORM not in {"cpu", "gpu", "cuda"}:
    raise ValueError("CHURN_TABFM_JAX_PLATFORM must be one of: cpu, gpu, cuda")

# The JAX GPU checkpoint restore currently OOMs on the RTX 4060 Laptop GPU's 8 GB VRAM.
# Keep the default CPU setting unless you are deliberately retesting GPU memory behavior.
if JAX_EXECUTION_PLATFORM == "cpu":
    os.environ["JAX_PLATFORMS"] = "cpu"

os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.setdefault("XLA_PYTHON_CLIENT_ALLOCATOR", "platform")
os.environ.setdefault("TF_GPU_ALLOCATOR", "cuda_malloc_async")
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")
print(f"Requested JAX execution platform: {JAX_EXECUTION_PLATFORM}")


In [ ]:
import json
import os
import sys
from pathlib import Path

import jax
import jax.numpy as jnp
import mlflow
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from dotenv import load_dotenv
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from tabfm import TabFMClassifier
import tabfm.src.jax.tabfm_v1_0_0 as tabfm_v1_0_0_jax

def find_project_root():
    for directory in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (directory / "src/churn_ml").exists():
            return directory
    raise FileNotFoundError("Could not find project root containing src/churn_ml.")

PROJECT_ROOT = find_project_root()
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from churn_ml.models.evaluate_model import classification_metrics

In [ ]:
RANDOM_STATE = 42
TARGET_COLUMN = "Churn Value"
CHURN_THRESHOLD = None  # Set a value from 0 to 1 to override the training churn-rate threshold.
TABFM_N_ESTIMATORS = 4
TABFM_BATCH_SIZE = 1
TABFM_MAX_CONTEXT_ROWS = 1024
# Conservative fallback for 8 GB GPUs if JAX still OOMs:
# TABFM_N_ESTIMATORS = 2
# TABFM_MAX_CONTEXT_ROWS = 512
TABFM_REPO_ID = "google/tabfm-1.0.0-jax"
TABFM_MODEL_TYPE = "classification"
JAX_COL_ATTENTION_IMPL = "flash"
JAX_ROW_ATTENTION_IMPL = "jax"
JAX_ICL_ATTENTION_IMPL = "flash"
JAX_DTYPE = jnp.bfloat16

def find_project_file(relative_path):
    for directory in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        candidate = directory / relative_path
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Could not find {relative_path} from {Path.cwd()} or its parent directories.")

def positive_class_probability(classifier, X):
    class_labels = list(classifier.classes_)
    if 1 not in class_labels:
        raise ValueError(f"Expected positive class label 1 in {class_labels}.")
    return classifier.predict_proba(X)[:, class_labels.index(1)]

DATA_PATH = find_project_file(Path("data/processed/prediction_df_xgboost.csv"))
VALUE_DATA_PATH = find_project_file(Path("data/raw/Telco_customer_churn.csv"))
load_dotenv(PROJECT_ROOT / ".env", override=False)
HF_TOKEN_CONFIGURED = bool(os.getenv("HF_TOKEN") or os.getenv("HUGGING_FACE_HUB_TOKEN"))
MLFLOW_EXPERIMENT_NAME = "telco-churn-modeling"
mlflow.set_tracking_uri("sqlite:///" + (PROJECT_ROOT / "mlflow.db").as_posix())
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)
TABFM_CONFIG_PATH = PROJECT_ROOT / "artifacts/models/tabfm_jax_config.json"

jax_devices = jax.devices()
jax_backend = jax.default_backend()
print(f"JAX version: {jax.__version__}")
print(f"JAX backend: {jax_backend}")
print(f"JAX devices: {jax_devices}")
print(f"Hugging Face token configured: {HF_TOKEN_CONFIGURED}")
if JAX_EXECUTION_PLATFORM == "cpu":
    print("JAX is intentionally forced to CPU for this notebook. Use 04a for GPU-backed TabFM comparison.")
elif jax_backend != "gpu":
    print("Warning: JAX is not using the GPU backend. In native Windows this is expected; use WSL2 for CUDA.")

In [ ]:
model_df = pd.read_csv(DATA_PATH)
value_df = pd.read_csv(VALUE_DATA_PATH, usecols=["CustomerID", TARGET_COLUMN, "CLTV"])
assert model_df.columns[-1] == TARGET_COLUMN, "The target must be the final column."
X = model_df.drop(columns=TARGET_COLUMN)
y = model_df[TARGET_COLUMN]
if len(value_df) != len(model_df) or not value_df[TARGET_COLUMN].equals(y):
    raise ValueError("Raw CLTV values are not aligned with the processed modeling data.")

customer_ltv = value_df.set_index("CustomerID")["CLTV"].rename("predicted_ltv_if_retained")
customer_ids = value_df["CustomerID"]
X_train, X_test, y_train, y_test, ltv_train, ltv_test, customer_id_train, customer_id_test = train_test_split(
    X, y, customer_ltv, customer_ids, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
decision_threshold = float(y_train.mean()) if CHURN_THRESHOLD is None else float(CHURN_THRESHOLD)
if not 0 < decision_threshold < 1:
    raise ValueError("CHURN_THRESHOLD must be between 0 and 1.")

print(f"Training rows: {len(X_train):,}; test rows: {len(X_test):,}")
print(f"Prediction threshold: {decision_threshold:.1%}")

## GPU memory note

The JAX/Orbax TabFM checkpoint restore has failed on this machine's 8 GB RTX 4060 Laptop GPU before training begins, even with conservative batch and context settings. That makes the JAX GPU path unsuitable for the main model comparison in this project.

This notebook defaults to CPU via `JAX_PLATFORMS=cpu`, set before importing JAX. To deliberately retest the GPU path, start a fresh WSL kernel with `CHURN_TABFM_JAX_PLATFORM=gpu`, then rerun from the top. If you see `RESOURCE_EXHAUSTED`, `CUDA_ERROR_OUT_OF_MEMORY`, or `CUDA_ERROR_NOT_READY`, restart the kernel and return to the CPU default.


## Zero-shot TabFM JAX inference

TabFM is loaded with the JAX backend. The default settings prioritize a reliable run over speed: CPU execution, `TABFM_N_ESTIMATORS = 4`, and `TABFM_MAX_CONTEXT_ROWS = 1024`. Use the PyTorch notebook, `04a_tabFM_modeling.ipynb`, for the GPU-backed TabFM candidate used in model selection.


In [ ]:
try:
    tabfm_backbone = tabfm_v1_0_0_jax.load(
        model_type=TABFM_MODEL_TYPE,
        col_attention_impl=JAX_COL_ATTENTION_IMPL,
        row_attention_impl=JAX_ROW_ATTENTION_IMPL,
        icl_attention_impl=JAX_ICL_ATTENTION_IMPL,
        dtype=JAX_DTYPE,
    )
except Exception as exc:
    message = str(exc)
    if jax.default_backend() == "gpu" and any(token in message for token in ["RESOURCE_EXHAUSTED", "Out of memory", "CUDA_ERROR_OUT_OF_MEMORY"]):
        raise RuntimeError(
            "TabFM JAX ran out of GPU memory while restoring the checkpoint. "
            "Set CHURN_TABFM_JAX_PLATFORM=cpu, restart the WSL kernel, and rerun this notebook. "
            "Use 04a_tabFM_modeling.ipynb for the GPU-backed TabFM model comparison."
        ) from exc
    raise

tabfm_model = TabFMClassifier(
    model=tabfm_backbone,
    n_estimators=TABFM_N_ESTIMATORS,
    batch_size=TABFM_BATCH_SIZE,
    max_num_rows=TABFM_MAX_CONTEXT_ROWS,
    random_state=RANDOM_STATE,
    verbose=True,
)
tabfm_model.fit(X_train, y_train)
print("TabFM JAX fit complete.")


In [ ]:
y_proba = positive_class_probability(tabfm_model, X_test)
y_pred = (y_proba >= decision_threshold).astype(int)

tabfm_metrics = pd.Series(classification_metrics(y_test, y_pred, y_proba), name="tabfm_jax")
display(tabfm_metrics.to_frame())

TABFM_CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)
tabfm_config = {
    "model": "tabfm",
    "backend": "jax",
    "checkpoint": TABFM_REPO_ID,
    "jax_version": jax.__version__,
    "jax_backend": jax_backend,
    "jax_execution_platform": JAX_EXECUTION_PLATFORM,
    "jax_devices": [str(device) for device in jax_devices],
    "jax_dtype": str(JAX_DTYPE),
    "col_attention_impl": JAX_COL_ATTENTION_IMPL,
    "row_attention_impl": JAX_ROW_ATTENTION_IMPL,
    "icl_attention_impl": JAX_ICL_ATTENTION_IMPL,
    "n_estimators": TABFM_N_ESTIMATORS,
    "batch_size": TABFM_BATCH_SIZE,
    "max_context_rows": TABFM_MAX_CONTEXT_ROWS,
    "random_state": RANDOM_STATE,
    "holdout_metrics": tabfm_metrics.to_dict(),
}
with TABFM_CONFIG_PATH.open("w", encoding="utf-8") as file:
    json.dump(tabfm_config, file, indent=2)
print(f"Saved TabFM JAX configuration and holdout metrics to {TABFM_CONFIG_PATH}")

## Holdout confusion matrix

The classification threshold defaults to the training-set churn rate for consistency with the other modeling notebooks.

In [ ]:
confusion = confusion_matrix(y_test, y_pred, labels=[0, 1])
fig = go.Figure(go.Heatmap(
    z=confusion,
    x=["Predicted: no churn", "Predicted: churn"],
    y=["Actual: no churn", "Actual: churn"],
    colorscale="Blues",
    text=confusion,
    texttemplate="%{text}",
    colorbar={"title": "Customers"},
))
fig.update_layout(title=f"TabFM JAX Confusion Matrix (threshold = {decision_threshold:.1%})")
fig.show()

# Expected Value of Retention Targeting

This evaluation retrieves the raw dataset's `CLTV` value for every holdout-test customer and treats it as that customer's predicted lifetime value if retained. `CLTV` is deliberately not a churn-model feature; it is used only after prediction to prioritize outreach.

In [ ]:
OUTREACH_COST = 20
OFFER_COST = 500
RETENTION_UPLIFT = 0.10
OFFER_ACCEPTANCE_RATE = 0.40
TARGET_COUNT = 100

targeting_candidates = pd.DataFrame({
    "CustomerID": customer_id_test.to_numpy(),
    "predicted_churn_probability": y_proba,
    "predicted_ltv_if_retained": ltv_test.to_numpy(),
})
targeting_candidates["expected_value_before_cost"] = (
    targeting_candidates["predicted_churn_probability"]
    * RETENTION_UPLIFT
    * targeting_candidates["predicted_ltv_if_retained"]
)
targeting_candidates["expected_offer_cost"] = OFFER_COST * OFFER_ACCEPTANCE_RATE
targeting_candidates["campaign_cost"] = OUTREACH_COST + targeting_candidates["expected_offer_cost"]
targeting_candidates["expected_net_value"] = (
    targeting_candidates["expected_value_before_cost"]
    - targeting_candidates["campaign_cost"]
)

top_100_targets = (
    targeting_candidates
    .sort_values("expected_net_value", ascending=False)
    .head(TARGET_COUNT)
    .reset_index(drop=True)
)

targeting_summary = pd.DataFrame({
    "customers_targeted": [len(top_100_targets)],
    "expected_value_before_cost": [top_100_targets["expected_value_before_cost"].sum()],
    "outreach_cost": [OUTREACH_COST * len(top_100_targets)],
    "expected_offer_cost": [top_100_targets["expected_offer_cost"].sum()],
    "campaign_cost": [top_100_targets["campaign_cost"].sum()],
    "expected_net_value": [top_100_targets["expected_net_value"].sum()],
})
display(targeting_summary.style.format({
    "expected_value_before_cost": "${:,.2f}",
    "outreach_cost": "${:,.2f}",
    "expected_offer_cost": "${:,.2f}",
    "campaign_cost": "${:,.2f}",
    "expected_net_value": "${:,.2f}",
}))
top_100_targets.style.format({
    "predicted_churn_probability": "{:.1%}",
    "predicted_ltv_if_retained": "${:,.0f}",
    "expected_value_before_cost": "${:,.2f}",
    "expected_offer_cost": "${:,.2f}",
    "campaign_cost": "${:,.2f}",
    "expected_net_value": "${:,.2f}",
})

## MLflow tracking

This section records the lightweight experiment evidence for the JAX backend: backend/device metadata, holdout metrics, and retention-targeting outputs. It does not log checkpoint weights.

In [ ]:
with mlflow.start_run(run_name="04_tabfm_jax_modeling"):
    mlflow.set_tags({
        "notebook": "04b_tabFM_JAX_modeling.ipynb",
        "model_family": "tabfm",
        "backend": "jax",
        "stage": "candidate_modeling",
    })
    mlflow.log_params({
        "random_state": RANDOM_STATE,
        "target_column": TARGET_COLUMN,
        "data_path": str(DATA_PATH.relative_to(PROJECT_ROOT)),
        "threshold_policy": "training_churn_rate" if CHURN_THRESHOLD is None else "manual",
        "decision_threshold": decision_threshold,
        "checkpoint": TABFM_REPO_ID,
        "jax_version": jax.__version__,
        "jax_backend": jax_backend,
        "jax_execution_platform": JAX_EXECUTION_PLATFORM,
        "jax_dtype": str(JAX_DTYPE),
        "col_attention_impl": JAX_COL_ATTENTION_IMPL,
        "row_attention_impl": JAX_ROW_ATTENTION_IMPL,
        "icl_attention_impl": JAX_ICL_ATTENTION_IMPL,
        "n_estimators": TABFM_N_ESTIMATORS,
        "batch_size": TABFM_BATCH_SIZE,
        "max_context_rows": TABFM_MAX_CONTEXT_ROWS,
    })
    mlflow.log_param("jax_devices", ", ".join(str(device) for device in jax_devices))
    mlflow.log_metrics({f"holdout_{key}": float(value) for key, value in tabfm_metrics.to_dict().items()})
    mlflow.log_metrics({
        "targeting_expected_net_value": float(targeting_summary.loc[0, "expected_net_value"]),
        "targeting_expected_value_before_cost": float(targeting_summary.loc[0, "expected_value_before_cost"]),
        "targeting_campaign_cost": float(targeting_summary.loc[0, "campaign_cost"]),
        "targeting_customers_targeted": int(targeting_summary.loc[0, "customers_targeted"]),
    })
    mlflow.log_dict(tabfm_config, "configs/tabfm_jax_config.json")
    mlflow.log_table(tabfm_metrics.reset_index().rename(columns={"index": "metric", "tabfm_jax": "value"}), "tables/holdout_metrics.json")
    mlflow.log_table(targeting_summary, "tables/targeting_summary.json")
    mlflow.log_table(top_100_targets, "tables/top_100_targets.json")

print(f"Logged MLflow run to {PROJECT_ROOT / 'mlflow.db'}")

## JAX notes

This notebook is intentionally separate from the PyTorch TabFM notebook because JAX GPU support is available here through WSL2, not native Windows. If JAX reports a CPU backend, switch to the WSL kernel/environment before interpreting runtime comparisons.